# Day 045 — Exercise 4: load

**What you'll build:** `load(session, records: list) -> int` — bulk-insert a list of clean dicts into the `sales` table using `session.add_all()` and return the count of rows loaded.

**Why it matters:** `session.add_all([Sale(**r) for r in records])` is more efficient than calling `session.add()` in a loop — it stages all objects in one operation before a single `commit()`. The function returns `int` (the count loaded) so `run_pipeline` can calculate how many records were skipped.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import csv
import io
from sqlalchemy import create_engine, String, Float, select, text
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, Session
from sqlalchemy.pool import StaticPool


class Base(DeclarativeBase):
    pass


class Sale(Base):
    __tablename__ = 'sales'
    id:       Mapped[int]   = mapped_column(primary_key=True)
    date:     Mapped[str]   = mapped_column(String(20))
    product:  Mapped[str]   = mapped_column(String(100))
    category: Mapped[str]   = mapped_column(String(50))
    amount:   Mapped[float] = mapped_column()
    region:   Mapped[str]   = mapped_column(String(50))

    def __repr__(self):
        return f'Sale(id={self.id}, product={self.product!r}, amount={self.amount})'


def setup_engine(url='sqlite:///:memory:'):
    engine = create_engine(
        url,
        connect_args={'check_same_thread': False},
        poolclass=StaticPool,
    )
    Base.metadata.create_all(engine)
    return engine


def extract(csv_text: str) -> list:
    reader = csv.DictReader(io.StringIO(csv_text))
    return list(reader)


def validate_record(record: dict) -> bool:
    required = ['date', 'product', 'amount']
    for field in required:
        if not record.get(field, '').strip():
            return False
    try:
        float(record['amount'])
    except (ValueError, TypeError):
        return False
    return True


def transform_record(record: dict) -> dict:
    return {
        'date':     record['date'].strip(),
        'product':  record['product'].strip(),
        'category': record.get('category', '').strip(),
        'amount':   round(float(record['amount']), 2),
        'region':   record.get('region', '').strip().title(),
    }


CSV_SOURCE = (
    'date,product,category,amount,region\n'
    '2024-01-15,Laptop,Electronics,999.99,East\n'
    '2024-01-16,Headphones,Electronics,149.99,West\n'
    '2024-01-17,Desk Chair,Furniture,349.00,East\n'
    '2024-01-18,,Furniture,199.00,North\n'
    '2024-01-19,Pen Set,Stationery,twelve,South\n'
    '2024-01-20,Monitor,Electronics,599.99,West\n'
    '2024-01-21,Keyboard,Electronics,79.99,East\n'
    '2024-01-22,Webcam,Electronics,,North\n'
    '2024-01-23,Lamp,Furniture,45.99,South\n'
    '2024-01-24,Notebook,Stationery,8.99,West\n'
)

engine  = setup_engine()
session = Session(engine)

# Pre-transform 3 clean records for use in checks
raw     = extract(CSV_SOURCE)
ready   = [transform_record(r) for r in raw if validate_record(r)][:3]

## Your Implementation

In [ ]:
def load(session, records: list) -> int:
    """
    Bulk-insert a list of clean dicts into the sales table.

    Steps:
    1. sales = [Sale(**r) for r in records]  — create ORM objects
    2. session.add_all(sales)                — stage all at once
    3. session.commit()                      — flush to DB
    4. return len(sales)
    """
    # TODO: sales = [Sale(**r) for r in records]
    # TODO: session.add_all(sales)
    # TODO: session.commit()
    # TODO: return len(sales)
    pass

## Check Your Work

In [ ]:
def _run_checks():
    total = 5
    passed = 0

    # Check 1: defined
    try:
        assert 'load' in globals()
        passed += 1; print('\u2705 Check 1: load is defined')
    except Exception as e:
        print(f'\u274c Check 1: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 2: returns an int
    try:
        count = load(session, ready)
        assert isinstance(count, int), \
            f'expected int, got {type(count).__name__}'
        passed += 1; print(f'\u2705 Check 2: returns int ({count})')
    except Exception as e:
        print(f'\u274c Check 2: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 3: returned count matches records passed
    try:
        assert count == len(ready) == 3, \
            f'expected 3, got {count}'
        passed += 1; print(f'\u2705 Check 3: count == {len(ready)} (records passed)')
    except Exception as e:
        print(f'\u274c Check 3: {e}')

    # Check 4: rows are in the database
    try:
        from sqlalchemy import select as sa_select
        all_sales = session.execute(sa_select(Sale)).scalars().all()
        assert len(all_sales) == 3, \
            f'expected 3 rows in DB, got {len(all_sales)}'
        passed += 1; print(f'\u2705 Check 4: {len(all_sales)} rows in DB')
    except Exception as e:
        print(f'\u274c Check 4: {e}')

    # Check 5: amount is stored as float
    try:
        first = session.execute(sa_select(Sale)).scalars().first()
        assert isinstance(first.amount, float), \
            f'amount should be float, got {type(first.amount).__name__}'
        passed += 1; print(f'\u2705 Check 5: amount stored as float ({first.amount})')
    except Exception as e:
        print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\U0001f389 Exercise complete!')
    print(f'\nScore: {passed}/{total}')


_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
def load(session, records: list) -> int:
    sales = [Sale(**r) for r in records]
    session.add_all(sales)
    session.commit()
    return len(sales)
```

</details>